# Fretwork — Any-Clip Audio → ASCII Tab (CAGED box version)

Self-contained pipeline: drop in **any** audio clip and get an ASCII tab. No GuitarSet
data, no held-out split, no evaluation harness — just inference.

**Three fixes baked in vs. the eval notebook:**

1. **Fret-span fix.** Chord grips are now hard-walled at **4 frets** (`MAX_CHORD_SPAN`),
   decoupled from the looser hand-reach tolerance used for melodic phrasing.
2. **CAGED positions.** Position selection is steered by key-derived **box anchors**
   (pentatonic CAGED boxes) instead of the learned GuitarSet position prior. For a clip
   in A minor the algorithm prefers box 1 at the 5th fret — exactly how *I Don't Trust
   Myself* is actually played.
3. **No more single-string climbs.** The Viterbi state is now the **hand position
   (box window)**, not the per-note fret. A run that drifts up one string would force the
   hand window to travel, which is penalized — so the solver keeps phrases inside one box
   and uses adjacent strings, like a real player. This fixes the funk output.

Run the cells top to bottom. Set your clip in the **last** config cell.

In [ ]:
# ============================================================
# Basic Pitch install/import (Colab Python 3.11/3.12 safe). Run ONCE.
# If not on Colab and these are already installed, this is a no-op-ish.
# ============================================================
import sys, subprocess, pkgutil, zipimport

def _run(cmd):
    print("$", " ".join(cmd))
    subprocess.check_call(cmd)

if not hasattr(pkgutil, "ImpImporter"):
    pkgutil.ImpImporter = zipimport.zipimporter

try:
    import librosa  # noqa
    from basic_pitch.inference import predict as basic_pitch_predict  # noqa
    print("Basic Pitch + librosa already available.")
except Exception:
    _run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip", "wheel", "setuptools==80.9.0"])
    _run([sys.executable, "-m", "pip", "install", "-q",
          "librosa>=0.10", "soundfile", "resampy==0.4.2", "onnxruntime", "pretty_midi", "mir_eval"])
    _run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "basic-pitch==0.4.0"])
    if not hasattr(pkgutil, "ImpImporter"):
        pkgutil.ImpImporter = zipimport.zipimporter
    import librosa  # noqa
    from basic_pitch.inference import predict as basic_pitch_predict  # noqa
    print("Installed and imported Basic Pitch + librosa.")

$ /usr/bin/python3 -m pip install -q --upgrade pip wheel setuptools==80.9.0
$ /usr/bin/python3 -m pip install -q librosa>=0.10 soundfile resampy==0.4.2 onnxruntime pretty_midi mir_eval
$ /usr/bin/python3 -m pip install -q --no-deps basic-pitch==0.4.0


/usr/local/lib/python3.12/dist-packages/resampy/filters.py:50: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


Installed and imported Basic Pitch + librosa.


In [ ]:
from pathlib import Path
from itertools import product
from collections import defaultdict
import math, re, warnings

import numpy as np
import pandas as pd
import librosa
import soundfile as sf
from basic_pitch.inference import predict as basic_pitch_predict

warnings.filterwarnings("ignore")

# Optional Colab Drive mount (harmless off-Colab).
IN_COLAB = Path("/content").exists()
if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as e:
        print("Drive mount skipped:", e)
print("Running in Colab:" , IN_COLAB)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Running in Colab: True


In [ ]:
# ============================================================
# CONFIG  (tweak the constants here to retune the three fixes)
# ============================================================

# --- Fretboard ---
MAX_FRET = 22
OPEN_STRING_MIDI = [40, 45, 50, 55, 59, 64]   # E2 A2 D3 G3 B3 E4
STRING_NAMES = ["low_E", "A", "D", "G", "B", "high_E"]
ONSET_TOLERANCE_SECONDS = 0.035
MAX_GROUP_CANDIDATES = 25

# --- FIX 1: fret span ---
# Hand-reach tolerance for a single finger across a melodic phrase (kept generous):
COMFORTABLE_SPAN = 5
MAX_REACHABLE_SPAN = 7
# Chord-grip span (the real fix). No one barres a 7-fret chord; cap the wall at 4.
COMFORTABLE_CHORD_SPAN = 3
MAX_CHORD_SPAN = 4
LARGE_JUMP_THRESHOLD = 5

# --- FIX 2 & 3: CAGED box anchoring ---
BOX_WINDOW = 4            # a hand position spans this many frets (e.g. anchor 5 -> frets 5..9)
SHIFT_FREE = 2           # repositioning the hand by <= this many frets is free
BOX_CENTER_COST   = 0.15  # mild pull toward the centre of the active window
BOX_OUTSIDE_COST  = 3.00  # strong penalty per fret a note sits OUTSIDE the active window
OPEN_OUT_OF_BOX_COST = 0.60  # using an open string while parked in a high box breaks the shape
BOX_OFFBOX_COST   = 0.60  # prefer window anchors that line up with a real pentatonic box
BOX_NONHOME_COST  = 1.00  # prefer the "home" box (root on the low E string, i.e. box 1)
BOX_LOWNECK_COST  = 0.04  # tiny tiebreak toward lower neck positions

WEIGHTS = {
    "playability": 0.80,   # span / awkwardness / duplicate-string
    "context":     0.30,   # in-key / in-chord softness
    "box_window":  1.00,   # how hard to respect the active CAGED window
    "hand_move":   0.70,   # cost of shifting the hand position between notes
}

# --- Basic Pitch note detection ---
BASIC_PITCH_AMPLITUDE_THRESHOLD = 0.30
BASIC_PITCH_MIN_MIDI = 40
BASIC_PITCH_MAX_MIDI = 79

# --- Output dir ---
OUTPUT_DIR = (Path("/content/drive/MyDrive/Capstone/outputs/caged_audio_to_tab")
              if IN_COLAB else Path("fretwork_outputs"))
try:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
except Exception:
    OUTPUT_DIR = Path("fretwork_outputs"); OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
BASIC_PITCH_CACHE_DIR = OUTPUT_DIR / "basic_pitch_note_cache"
BASIC_PITCH_CACHE_DIR.mkdir(parents=True, exist_ok=True)
print("Output dir:", OUTPUT_DIR.resolve())

Output dir: /content/drive/MyDrive/Capstone/outputs/caged_audio_to_tab


In [ ]:
# ============================================================
# Fretboard layout + MIDI -> positions
# ============================================================
def build_fretboard(open_string_midi=OPEN_STRING_MIDI, max_fret=MAX_FRET):
    rows = []
    for s, open_midi in enumerate(open_string_midi):
        for fret in range(max_fret + 1):
            midi = open_midi + fret
            rows.append({"string": s, "string_name": STRING_NAMES[s], "fret": fret,
                         "midi": midi, "pitch_class": midi % 12})
    return pd.DataFrame(rows)

fretboard_df = build_fretboard()
MIDI_TO_POSITIONS = defaultdict(list)
for row in fretboard_df.to_dict("records"):
    MIDI_TO_POSITIONS[int(row["midi"])].append({
        "string": int(row["string"]), "string_name": row["string_name"],
        "fret": int(row["fret"]), "midi": int(row["midi"]),
        "pitch_class": int(row["pitch_class"]),
    })

def get_possible_positions(midi_note, max_fret=MAX_FRET):
    midi_note = int(round(midi_note))
    return [p for p in MIDI_TO_POSITIONS.get(midi_note, []) if 0 <= p["fret"] <= max_fret]

print("Fretboard positions for MIDI 52 (E3):", [(p["string_name"], p["fret"]) for p in get_possible_positions(52)])

Fretboard positions for MIDI 52 (E3): [('low_E', 12), ('A', 7), ('D', 2)]


In [ ]:
# ============================================================
# Pitch classes, scales, key database
# ============================================================
PITCH_CLASS_NAMES_SHARP = ["C","C#","D","D#","E","F","F#","G","G#","A","A#","B"]
NOTE_TO_PC = {n: i for i, n in enumerate(PITCH_CLASS_NAMES_SHARP)}
NOTE_TO_PC.update({"Db":1,"Eb":3,"Gb":6,"Ab":8,"Bb":10})
PC_TO_NOTE = {i: n for i, n in enumerate(PITCH_CLASS_NAMES_SHARP)}

MAJOR_STEPS = [2,2,1,2,2,2,1]
MINOR_STEPS = [2,1,2,2,1,2,2]

def derive_scale(root_pc, mode="major"):
    steps = MAJOR_STEPS if mode == "major" else MINOR_STEPS
    pcs, cur = [root_pc], root_pc
    for step in steps[:-1]:
        cur = (cur + step) % 12; pcs.append(cur)
    return pcs

def build_key_database():
    rows = []
    for root_name, root_pc in NOTE_TO_PC.items():
        if "b" in root_name:  # keep one spelling per pc
            continue
        for mode in ["major", "minor"]:
            rows.append({"key": f"{root_name} {mode}", "root": root_name,
                         "root_pc": root_pc, "mode": mode,
                         "scale_pcs": derive_scale(root_pc, mode)})
    return pd.DataFrame(rows)

key_db = build_key_database()

def get_key_info(key_label):
    if key_label is None:
        return None
    s = str(key_label).replace(":", " ").strip()
    toks = s.split()
    if len(toks) == 1 and toks[0] in NOTE_TO_PC:
        s = f"{toks[0]} major"
    m = key_db[key_db["key"] == s]
    return m.iloc[0].to_dict() if len(m) else None

def parse_key(key_label):
    info = get_key_info(key_label)
    if info is None:
        return None
    return {"root_pc": int(info["root_pc"]), "mode": info["mode"],
            "scale_pcs": set(info["scale_pcs"])}

print("A minor ->", parse_key("A minor"))

A minor -> {'root_pc': 9, 'mode': 'minor', 'scale_pcs': {0, 2, 4, 5, 7, 9, 11}}


In [ ]:
# ============================================================
# Chord knowledge (used only as soft context)
# ============================================================
CHORD_INTERVALS = {"maj":[0,4,7],"min":[0,3,7],"dim":[0,3,6],"aug":[0,4,8],
                   "7":[0,4,7,10],"maj7":[0,4,7,11],"min7":[0,3,7,10],
                   "sus4":[0,5,7],"sus2":[0,2,7],"5":[0,7]}
QUALITY_ALIASES = {"M":"maj","major":"maj","":"maj","m":"min","minor":"min","dom7":"7"}

def normalize_quality(q):
    if q is None: return "maj"
    return QUALITY_ALIASES.get(str(q).strip(), str(q).strip())

def chord_tones(root_pc, quality="maj"):
    quality = normalize_quality(quality)
    iv = CHORD_INTERVALS.get(quality, CHORD_INTERVALS["maj"])
    return sorted({(root_pc + i) % 12 for i in iv})

def parse_chord_symbol(symbol):
    if symbol is None: return None
    s = str(symbol).strip().split("/")[0]
    if s in ["N","X","nan","None",""]: return None
    if ":" in s:
        root, qual = s.split(":", 1)
    else:
        m = re.match(r"^([A-G](?:#|b)?)(.*)$", s)
        if not m: return None
        root, qual = m.group(1), m.group(2)
    if root not in NOTE_TO_PC: return None
    qual = normalize_quality(qual)
    return {"root": root, "root_pc": NOTE_TO_PC[root], "quality": qual,
            "tones": chord_tones(NOTE_TO_PC[root], qual)}

In [ ]:
# ============================================================
# CAGED position boxes  (FIX 2 & FIX 3 core)
# ------------------------------------------------------------
# A "hand position" is a 4-fret window [anchor, anchor+BOX_WINDOW]. For the
# detected key we mark which windows line up with real pentatonic CAGED boxes,
# and which one is the "home" box (root on the low E string = box 1). The
# Viterbi later treats the anchor as its state, so a phrase commits to a box
# and only pays to shift the hand when the melody genuinely moves.
# ============================================================
PENTATONIC = {"major": [0,2,4,7,9], "minor": [0,3,5,7,10]}
LOW_E_PC = OPEN_STRING_MIDI[0] % 12   # 4 (E)

def box_anchors_for_key(key, max_fret=MAX_FRET, window=BOX_WINDOW):
    """List of candidate hand-position windows with a key-fit cost each."""
    anchors_range = range(0, max_fret - window + 1)
    if key is None:
        # No key -> every window, only a faint low-neck tiebreak. Compactness
        # still comes from the hand-move penalty in the Viterbi.
        return [{"anchor": a, "home": False, "key_cost": BOX_LOWNECK_COST * a}
                for a in anchors_range]

    r = key["root_pc"]
    penta = PENTATONIC.get(key["mode"], PENTATONIC["minor"])

    # Frets on the low E string that are pentatonic tones => box start anchors.
    box_anchor_frets = set()
    for deg in penta:
        f = (deg + (r - LOW_E_PC)) % 12
        while f <= max_fret - 1:
            box_anchor_frets.add(f); f += 12
    # Home box (box 1): root sits on the low E string.
    home_anchors = set()
    h = (r - LOW_E_PC) % 12
    while h <= max_fret - 1:
        home_anchors.add(h); h += 12

    out = []
    for a in anchors_range:
        d_box = min((abs(a - b) for b in box_anchor_frets), default=0)
        is_home = a in home_anchors
        key_cost = (BOX_OFFBOX_COST * d_box
                    + (0.0 if is_home else BOX_NONHOME_COST)
                    + BOX_LOWNECK_COST * a)
        out.append({"anchor": a, "home": is_home, "key_cost": key_cost})
    return out

def position_window_cost(p, anchor, window=BOX_WINDOW):
    """Cost of placing one note at position p given the active hand window."""
    f = p["fret"]
    if f == 0:
        return 0.0 if anchor <= 2 else OPEN_OUT_OF_BOX_COST
    if anchor <= f <= anchor + window:
        return BOX_CENTER_COST * abs(f - (anchor + window / 2.0))
    dist = (anchor - f) if f < anchor else (f - (anchor + window))
    return BOX_OUTSIDE_COST * dist

def candidate_window_cost(cand, anchor):
    return sum(position_window_cost(p, anchor) for p in cand["positions"])

In [ ]:
# ============================================================
# Playability scoring  (FIX 1: chord span hard-walled at MAX_CHORD_SPAN)
# ============================================================
def estimate_hand_position_from_frets(frets):
    fretted = [f for f in frets if f > 0]
    return 0 if not fretted else int(round(np.median(fretted)))

def group_span(frets):
    fretted = [f for f in frets if f > 0]   # open strings never count toward span
    return 0 if len(fretted) <= 1 else max(fretted) - min(fretted)

def awkward_fingering_penalty(position, hand_center):
    fret = position["fret"]
    if fret == 0:
        return 0.0
    d = abs(fret - hand_center)
    if d <= 2: return 0.0
    if d <= COMFORTABLE_SPAN:  return 0.5 * (d - 2)
    if d <= MAX_REACHABLE_SPAN: return 2.0 + (d - COMFORTABLE_SPAN)
    return 10.0 + 2.0 * (d - MAX_REACHABLE_SPAN)

def group_playability_cost(group_positions):
    if not group_positions:
        return 0.0
    strings = [p["string"] for p in group_positions]
    frets   = [p["fret"]   for p in group_positions]
    fretted = [f for f in frets if f > 0]
    if len(strings) != len(set(strings)):       # two notes on one string = impossible
        return float("inf")
    cost = 0.0
    span = group_span(frets)
    if span > COMFORTABLE_CHORD_SPAN:
        cost += 2.0 * (span - COMFORTABLE_CHORD_SPAN)
    if span > MAX_CHORD_SPAN:                    # hard wall at 4 frets
        cost += 25.0 * (span - MAX_CHORD_SPAN)
    if fretted and min(fretted) <= 2 and max(fretted) >= 9:
        cost += 8.0
    if len(strings) >= 2:
        ss = max(strings) - min(strings)
        if ss > 4 and len(strings) <= 3:
            cost += 1.5 * (ss - 4)
    hand_center = estimate_hand_position_from_frets(frets)
    cost += sum(awkward_fingering_penalty(p, hand_center) for p in group_positions)
    if any(f == 0 for f in frets) and fretted and max(fretted) > 7:
        cost += 3.0
    return cost

def context_cost(group_notes, group_positions):
    cost = 0.0
    for n, p in zip(group_notes, group_positions):
        if n.get("in_chord") is False: cost += 0.15
        if n.get("in_key")   is False: cost += 0.10
    return cost

In [ ]:
# ============================================================
# Onset grouping + per-group candidate generation
# ============================================================
def group_notes_by_onset(notes, tolerance=ONSET_TOLERANCE_SECONDS):
    if not notes:
        return []
    ns = sorted(notes, key=lambda x: (x["start"], x["midi"]))
    groups, cur, start = [], [ns[0]], ns[0]["start"]
    for n in ns[1:]:
        if abs(n["start"] - start) <= tolerance:
            cur.append(n)
        else:
            groups.append(cur); cur = [n]; start = n["start"]
    groups.append(cur)
    return groups

def enrich_candidate(c):
    frets = [p["fret"] for p in c["positions"]]
    c["center"] = estimate_hand_position_from_frets(frets)
    return c

def candidate_groups(group_notes, max_candidates=MAX_GROUP_CANDIDATES):
    """Valid string/fret combos for an onset group, scored by playability+context.
    No learned prior here -- the CAGED window does the positional steering."""
    position_lists = []
    for n in group_notes:
        pos = get_possible_positions(n["midi"])
        if not pos:
            return []
        position_lists.append(pos)
    candidates = []
    for combo in product(*position_lists):
        combo = list(combo)
        if len(combo) > 1 and len({p["string"] for p in combo}) != len(combo):
            continue
        play = group_playability_cost(combo)
        if not math.isfinite(play):
            continue
        base = WEIGHTS["playability"] * play + WEIGHTS["context"] * context_cost(group_notes, combo)
        candidates.append(enrich_candidate({"positions": combo, "base_cost": float(base)}))
    if not candidates:   # extreme fallback: keep something playable
        for combo in product(*position_lists):
            combo = list(combo)
            if len(combo) > 1 and len({p["string"] for p in combo}) != len(combo):
                continue
            candidates.append(enrich_candidate({"positions": combo, "base_cost": 50.0}))
    return sorted(candidates, key=lambda c: c["base_cost"])[:max_candidates]

In [ ]:
# ============================================================
# Box-anchored Viterbi assignment  (the method that fixes all three issues)
# ------------------------------------------------------------
# State = hand-position window (anchor). For each onset group and each anchor we
# take the best in-window candidate as the emission. Transitions penalise moving
# the hand. Result: phrases stay inside a CAGED box and only shift when the music
# really moves, instead of climbing one string or jumping to a random region.
# ============================================================
def assign_caged_box(notes, weights=WEIGHTS, key=None):
    groups = group_notes_by_onset(notes)
    if not groups:
        return []
    all_cands = [candidate_groups(g) for g in groups]
    if any(len(c) == 0 for c in all_cands):
        raise ValueError("An onset group had no playable candidate positions.")

    anchors = box_anchors_for_key(key)
    A = len(anchors)
    anchor_fret = np.array([a["anchor"] for a in anchors], dtype=float)
    anchor_keyc = np.array([a["key_cost"] for a in anchors], dtype=float)
    n = len(groups)

    # Emission: emit_cost[i, j] and which candidate produced it.
    emit_cost = np.empty((n, A), dtype=float)
    emit_cand = [[0] * A for _ in range(n)]
    for i, cands in enumerate(all_cands):
        for j, anc in enumerate(anchors):
            a = anc["anchor"]
            best, best_ci = None, 0
            for ci, c in enumerate(cands):
                tot = c["base_cost"] + weights["box_window"] * candidate_window_cost(c, a)
                if best is None or tot < best:
                    best, best_ci = tot, ci
            emit_cost[i, j] = best + anc["key_cost"]
            emit_cand[i][j] = best_ci

    # Transition between hand positions: free up to SHIFT_FREE frets, then linear.
    delta = np.abs(anchor_fret[:, None] - anchor_fret[None, :])
    shift = np.maximum(delta - SHIFT_FREE, 0.0)
    big = np.maximum(delta - LARGE_JUMP_THRESHOLD, 0.0)
    trans = weights["hand_move"] * shift + 0.5 * big**2

    dp = np.empty((n, A)); back = np.zeros((n, A), dtype=int)
    dp[0] = emit_cost[0]; back[0] = -1
    for i in range(1, n):
        scores = dp[i - 1][:, None] + trans + emit_cost[i][None, :]
        back[i] = np.argmin(scores, axis=0)
        dp[i] = scores[back[i], np.arange(A)]

    j = int(np.argmin(dp[-1]))
    chosen = [j]
    for i in range(n - 1, 0, -1):
        j = int(back[i][j]); chosen.append(j)
    chosen = list(reversed(chosen))

    pred = []
    for i, (g, cands) in enumerate(zip(groups, all_cands)):
        aj = chosen[i]
        c = cands[emit_cand[i][aj]]
        for note, p in zip(g, c["positions"]):
            row = dict(note)
            row.update({"pred_string": p["string"], "pred_fret": p["fret"],
                        "method": "caged_box", "anchor": anchors[aj]["anchor"]})
            pred.append(row)
    return sorted(pred, key=lambda x: (x["start"], x["midi"]))

def assign_baseline_lowest_fret(notes):
    """Commercial-tool-style baseline: always the lowest fret. For comparison only."""
    out = []
    for n in sorted(notes, key=lambda x: (x["start"], x["midi"])):
        pos = get_possible_positions(n["midi"])
        if not pos:
            continue
        p = min(pos, key=lambda p: (p["fret"], p["string"]))
        row = dict(n); row.update({"pred_string": p["string"], "pred_fret": p["fret"],
                                   "method": "lowest_fret"})
        out.append(row)
    return out

In [ ]:
# ============================================================
# Audio front-end: Basic Pitch notes + key + rough chords + beats
# ============================================================
def midi_to_note_name_simple(midi):
    names = ["C","C#","D","D#","E","F","F#","G","G#","A","A#","B"]
    midi = int(round(midi)); return f"{names[midi % 12]}{midi // 12 - 1}"

def drop_octave_harmonics(notes, tol=ONSET_TOLERANCE_SECONDS):
    groups = group_notes_by_onset(notes)
    kept = []
    for g in groups:
        midis = {n['midi'] for n in g}
        for n in g:
            harm = n['midi'] - 12 if (n['midi'] - 12) in midis else (
                   n['midi'] - 24 if (n['midi'] - 24) in midis else None)
            if harm is not None:
                amp_n   = n.get('amplitude') or 1.0
                amp_low = max((m.get('amplitude') or 1.0) for m in g if m['midi'] == harm)
                if amp_n <= amp_low:
                    continue  # drop: octave harmonic of a louder lower note
            kept.append(n)
    return sorted(kept, key=lambda x: (x['start'], x['midi']))

def _bp_cache_path(audio_path):
    return BASIC_PITCH_CACHE_DIR / f"{Path(audio_path).stem}_bp_notes.csv"

def run_basic_pitch_notes(audio_path, use_cache=True):
    audio_path = Path(audio_path)
    cache = _bp_cache_path(audio_path)
    if use_cache and cache.exists():
        return pd.read_csv(cache).to_dict("records")
    print("Running Basic Pitch on:", audio_path.name)
    _, _, note_events = basic_pitch_predict(str(audio_path))
    notes = []
    for ev in note_events:
        start, end, pitch_midi, amplitude = ev[0], ev[1], ev[2], ev[3]
        if float(amplitude) < BASIC_PITCH_AMPLITUDE_THRESHOLD:
            continue
        midi = int(round(float(pitch_midi)))
        if midi < BASIC_PITCH_MIN_MIDI or midi > BASIC_PITCH_MAX_MIDI:
            continue
        notes.append({"start": float(start), "duration": float(end - start), "midi": midi,
                      "pitch_class": midi % 12, "note_name": midi_to_note_name_simple(midi),
                      "amplitude": float(amplitude)})
    notes = sorted(notes, key=lambda n: (n["start"], n["midi"]))
    pd.DataFrame(notes).to_csv(cache, index=False)
    return notes

_MAJOR_PROFILE = np.array([6.35,2.23,3.48,2.33,4.38,4.09,2.52,5.19,2.39,3.66,2.29,2.88])
_MINOR_PROFILE = np.array([6.33,2.68,3.52,5.38,2.60,3.53,2.54,4.75,3.98,2.69,3.34,3.17])

def detect_key_from_audio(audio_path):
    y, sr = librosa.load(str(audio_path), sr=None, mono=True)
    chroma = np.nan_to_num(np.mean(librosa.feature.chroma_cqt(y=y, sr=sr), axis=1))
    scores = []
    for tonic in range(12):
        scores.append((f"{PC_TO_NOTE[tonic]} major", np.corrcoef(chroma, np.roll(_MAJOR_PROFILE, tonic))[0,1]))
        scores.append((f"{PC_TO_NOTE[tonic]} minor", np.corrcoef(chroma, np.roll(_MINOR_PROFILE, tonic))[0,1]))
    scores.sort(key=lambda x: x[1], reverse=True)
    return {"key": scores[0][0], "score": float(scores[0][1]), "top3": scores[:3]}

_CHORD_TEMPLATES = []
for root in range(12):
    _CHORD_TEMPLATES += [
        {"label": PC_TO_NOTE[root],        "root": root, "quality": "maj", "tones": {(root+x)%12 for x in [0,4,7]}},
        {"label": PC_TO_NOTE[root] + "m",  "root": root, "quality": "min", "tones": {(root+x)%12 for x in [0,3,7]}},
        {"label": PC_TO_NOTE[root] + "7",  "root": root, "quality": "7",   "tones": {(root+x)%12 for x in [0,4,7,10]}},
    ]

def _best_chord_for_pcs(pcs):
    pcs = set(int(p) % 12 for p in pcs)
    if len(pcs) < 2:
        return None
    best_tuple, best = None, None
    for idx, t in enumerate(_CHORD_TEMPLATES):
        tones = t["tones"]
        score = len(pcs & tones) - 0.45*len(tones - pcs) - 0.25*len(pcs - tones) + (0.35 if t["root"] in pcs else 0.0)
        tup = (score, len(pcs & tones), -idx)
        if best_tuple is None or tup > best_tuple:
            best_tuple, best = tup, t
    return best if (best and best_tuple[1] >= 2) else None

def detect_chords_from_notes(notes, window=1.0, hop=0.5, min_notes=2):
    if not notes:
        return []
    max_t = max(float(n["start"]) + float(n.get("duration", 0.0) or 0.0) for n in notes)
    chords, cur, prev_label, t = [], None, None, 0.0
    while t <= max_t:
        t_end = t + window
        pcs = [int(n["pitch_class"]) for n in notes
               if float(n["start"]) < t_end and float(n["start"]) + float(n.get("duration",0.0) or 0.0) >= t]
        tpl = _best_chord_for_pcs(pcs) if len(pcs) >= min_notes else None
        label = tpl["label"] if tpl else None
        if label is not None:
            if cur is not None and label == prev_label:
                cur["end"] = t_end
            else:
                if cur is not None: chords.append(cur)
                cur = {"start": float(t), "end": float(t_end), "chord": label,
                       "parsed": {"root": tpl["root"], "quality": tpl["quality"], "tones": sorted(tpl["tones"])}}
                prev_label = label
        else:
            if cur is not None: chords.append(cur); cur = None
            prev_label = None
        t += hop
    if cur is not None: chords.append(cur)
    return chords

def estimate_beats_tempo(audio_path):
    try:
        y, sr = librosa.load(str(audio_path), sr=None, mono=True)
        tempo, beat_frames = librosa.beat.beat_track(y=y, sr=sr)
        beats = librosa.frames_to_time(beat_frames, sr=sr).tolist()
        return float(np.atleast_1d(tempo)[0]), beats
    except Exception as e:
        print("Beat tracking failed, falling back to fixed grid:", e)
        return None, []

def chord_at_time(chords, t):
    for c in chords:
        if float(c.get("start", 0.0)) <= t < float(c.get("end", c.get("start",0.0))):
            return c
    return None

def enrich_notes_with_context(notes, key_label, chords):
    key_info = get_key_info(key_label)
    scale = set(key_info["scale_pcs"]) if key_info else None
    out = []
    for n in notes:
        c = chord_at_time(chords, n["start"])
        parsed = c.get("parsed") if c else None
        row = dict(n)
        row["key_label"] = key_label
        row["in_key"] = None if scale is None else (n["pitch_class"] in scale)
        row["in_chord"] = None if parsed is None else (n["pitch_class"] in set(parsed["tones"]))
        out.append(row)
    return out

In [ ]:
# ============================================================
# ASCII tab rendering
# ============================================================
DISPLAY_TO_STRING = [5, 4, 3, 2, 1, 0]      # high e on top, low E on bottom
STRING_LABELS     = ["e", "B", "G", "D", "A", "E"]

def _ci(v):
    if v is None: return None
    try:
        if pd.isna(v): return None
    except TypeError: pass
    return int(round(float(v)))

def _cf(v, d=0.0):
    if v is None: return d
    try:
        if pd.isna(v): return d
    except TypeError: pass
    return float(v)

def _build_time_grid(beats, tempo, end_time, sub=2):
    beats = sorted([float(b) for b in (beats or []) if b is not None])
    if len(beats) >= 2:
        grid = []
        for i in range(len(beats) - 1):
            step = (beats[i+1] - beats[i]) / sub
            if step <= 0: continue
            for j in range(sub): grid.append(beats[i] + j*step)
        if grid:
            last = (beats[-1] - beats[-2]) / sub or 0.25
            while grid[-1] < end_time: grid.append(grid[-1] + last)
            return grid
    if tempo and float(tempo) > 0:
        step = 60.0/float(tempo)/sub
        return [i*step for i in range(int(end_time/step) + sub + 2)]
    return [i*0.25 for i in range(int(end_time/0.25) + 3)]

def render_ascii_tab(parsed, sub=2, beats_per_measure=4, measures_per_line=4, col_width=3, max_notes=None):
    notes = sorted(parsed.get("notes", []), key=lambda n: (n.get("start",0.0), n.get("midi",0)))
    if max_notes is not None: notes = notes[:max_notes]
    if not notes: return "(no notes)"
    end_time = max(_cf(n.get("start")) + _cf(n.get("duration"), 0.5) for n in notes) + 0.5
    grid = _build_time_grid(parsed.get("beats", []), parsed.get("tempo"), end_time, sub)
    n_cols = len(grid)
    cells = [[None]*n_cols for _ in range(6)]
    collisions = 0
    for note in notes:
        s, f = _ci(note.get("string")), _ci(note.get("fret"))
        if s is None or f is None or s not in DISPLAY_TO_STRING: continue
        col = min(range(n_cols), key=lambda i: abs(grid[i] - _cf(note.get("start"))))
        row = DISPLAY_TO_STRING.index(s)
        if cells[row][col] is not None: collisions += 1
        cells[row][col] = f
    def fmt(v):
        if v is None: return "-"*col_width
        s = str(v); return s[:col_width] if len(s) >= col_width else s + "-"*(col_width-len(s))
    formatted = [[fmt(cells[r][c]) for c in range(n_cols)] for r in range(6)]
    cpm = beats_per_measure*sub; cpl = cpm*measures_per_line
    lines = []
    if parsed.get("title"):
        lines += [str(parsed["title"]), "-"*min(len(str(parsed["title"])), 80)]
    for start in range(0, n_cols, cpl):
        end = min(start+cpl, n_cols)
        for row in range(6):
            parts = []
            for c in range(start, end):
                if c > start and (c-start) % cpm == 0: parts.append("|")
                parts.append(formatted[row][c])
            lines.append(f"{STRING_LABELS[row]}|{''.join(parts)}|")
        lines.append("")
    if collisions:
        lines.append(f"[note: {collisions} notes shared a grid cell; later note shown]")
    return "\n".join(lines)

def predictions_to_render_dict(pred_rows, beats=None, tempo=None, title=None):
    notes = []
    for r in pred_rows:
        s, f = _ci(r.get("pred_string")), _ci(r.get("pred_fret"))
        if s is None or f is None or not (0 <= f <= MAX_FRET): continue
        notes.append({"start": _cf(r.get("start")), "duration": _cf(r.get("duration"), 0.5),
                      "midi": _ci(r.get("midi")), "string": s, "fret": f})
    return {"title": title, "notes": notes, "beats": beats or [], "tempo": tempo}

def render_predicted_tab(pred_rows, beats=None, tempo=None, title=None, max_notes=200):
    return render_ascii_tab(predictions_to_render_dict(pred_rows, beats, tempo, title), max_notes=max_notes)

In [ ]:
# ============================================================
# Main entry point: any audio clip -> ASCII tab
# ============================================================
def audio_clip_to_ascii_tab(audio_path, max_notes=200, show_baseline=True,
                            key_override=None, save=True):
    audio_path = Path(audio_path)
    if not audio_path.exists():
        raise FileNotFoundError(f"Audio file not found: {audio_path}")

    bp_notes = drop_octave_harmonics(run_basic_pitch_notes(audio_path))
    if not bp_notes:
        print("No notes detected (try lowering BASIC_PITCH_AMPLITUDE_THRESHOLD)."); return None

    key_label = key_override or detect_key_from_audio(audio_path)["key"]
    chords = detect_chords_from_notes(bp_notes)
    enriched = enrich_notes_with_context(bp_notes, key_label, chords)
    key = parse_key(key_label)

    pred = assign_caged_box(enriched, key=key)
    tempo, beats = estimate_beats_tempo(audio_path)

    print("=" * 70)
    print(f"Clip:  {audio_path.name}")
    print(f"Notes detected: {len(bp_notes)}   |   Detected key: {key_label}"
          + ("" if key_override is None else "  (overridden)"))
    print(f"Estimated tempo: {None if tempo is None else round(tempo,1)} BPM")
    print("=" * 70)

    caged_tab = render_predicted_tab(pred, beats, tempo, max_notes=max_notes,
                                     title=f"CAGED-BOX TAB - {audio_path.stem} - key {key_label}")
    print("\n" + "#"*70 + "\n# CAGED-BOX (FIXED) TAB\n" + "#"*70)
    print(caged_tab)

    base_tab = None
    if show_baseline:
        base_pred = assign_baseline_lowest_fret(enriched)
        base_tab = render_predicted_tab(base_pred, beats, tempo, max_notes=max_notes,
                                        title=f"LOWEST-FRET BASELINE - {audio_path.stem}")
        print("\n" + "#"*70 + "\n# LOWEST-FRET BASELINE (for comparison)\n" + "#"*70)
        print(base_tab)

    if save:
        safe = re.sub(r"[^A-Za-z0-9_.-]+", "_", audio_path.stem)
        out = OUTPUT_DIR / f"{safe}__caged_box_tab.txt"
        out.write_text(caged_tab + ("\n\n" + base_tab if base_tab else ""))
        print("\nSaved:", out.resolve())

    return {"key": key_label, "tempo": tempo, "n_notes": len(bp_notes),
            "predictions": pred, "caged_tab": caged_tab, "baseline_tab": base_tab}

## Run it on your clip

1. Put an audio file somewhere the notebook can see it.
   - **Colab:** run the upload helper below, or set `INPUT_AUDIO_PATH` to a Drive path.
   - **Local:** set `INPUT_AUDIO_PATH` to the file path.
2. Run the final cell.

The detected key drives the CAGED boxes. If key detection is wrong for a clip (the
chroma estimator is only ~55% accurate), pass `key_override="A minor"` to
`audio_clip_to_ascii_tab(...)` to force it.

In [ ]:
# ------------------------------------------------------------
# SET YOUR CLIP HERE
# ------------------------------------------------------------
INPUT_AUDIO_PATH =  "/content/drive/MyDrive/Capstone/Audio/isThisIt.mp3"

# Colab upload helper (optional). Uncomment to pick a file from your computer.
# if IN_COLAB and not INPUT_AUDIO_PATH:
#     from google.colab import files
#     up = files.upload()
#     INPUT_AUDIO_PATH = str(Path.cwd() / list(up.keys())[0])

if INPUT_AUDIO_PATH:
    result = audio_clip_to_ascii_tab(
        INPUT_AUDIO_PATH,
        max_notes=200,
        show_baseline=True,     # also print the lowest-fret baseline to compare
        key_override=None,      # e.g. "A minor" to force the key
    )
else:
    print("Set INPUT_AUDIO_PATH above (or use the Colab upload helper), then re-run this cell.")

Running Basic Pitch on: isThisIt.mp3
Predicting MIDI for /content/drive/MyDrive/Capstone/Audio/isThisIt.mp3...
Clip:  isThisIt.mp3
Notes detected: 45   |   Detected key: E major
Estimated tempo: 156.2 BPM

######################################################################
# CAGED-BOX (FIXED) TAB
######################################################################
CAGED-BOX TAB - isThisIt - key E major
--------------------------------------
e|1-----5-----1-----5-----|0--5--5-----------5-----|1-----5-----1-----5-----|0-----5-----------5-----|
B|------------------------|------------3-----------|------------------------|------------3-----3-----|
G|------------------------|------------------------|------------------------|------------------------|
D|------------------------|------------------------|------------------3-----|------------------------|
A|------------------------|------------------0-----|------------------------|------------------------|
E|------------------------|--------

In [ ]:
for r in result['predictions']:
    print(r['start'], r['midi'], midi_to_note_name_simple(r['midi']), '->', STRING_NAMES[r['pred_string']], r['pred_fret'], 'anchor', r['anchor'])

1.172607709750567 57 A3 -> D 7 anchor 5
1.3467573696145123 59 B3 -> D 9 anchor 5
1.822766439909297 62 D4 -> G 7 anchor 5
2.4045492063492064 45 A2 -> low_E 5 anchor 5
2.4045492063492064 61 C#4 -> G 6 anchor 5
3.3681773242630384 45 A2 -> low_E 5 anchor 5
3.9486761904761902 45 A2 -> low_E 5 anchor 5
4.5652888888888885 42 F#2 -> low_E 2 anchor 2
5.006468027210884 57 A3 -> D 7 anchor 5
5.598576870748299 57 A3 -> D 7 anchor 5
5.935266213151928 54 F#3 -> A 9 anchor 5
6.319679365079365 52 E3 -> A 7 anchor 5
6.784078458049887 43 G2 -> low_E 3 anchor 3
7.399407256235828 52 E3 -> A 7 anchor 5
8.004409977324263 50 D3 -> A 5 anchor 5
8.433979138321995 43 G2 -> low_E 3 anchor 3
9.188627664399093 59 B3 -> D 9 anchor 5
9.502097052154197 62 D4 -> G 7 anchor 5
9.803956462585033 59 B3 -> D 9 anchor 5
9.944560090702948 61 C#4 -> G 6 anchor 5
10.281249433106575 59 B3 -> D 9 anchor 5
10.606328798185942 61 C#4 -> G 6 anchor 5
10.780478458049888 59 B3 -> D 9 anchor 5
11.012678004535148 57 A3 -> D 7 anchor 5
1

In [ ]:
def drop_octave_harmonics(notes, tol=ONSET_TOLERANCE_SECONDS):
    groups = group_notes_by_onset(notes)
    kept = []
    for g in groups:
        midis = {n['midi'] for n in g}
        for n in g:
            harm = n['midi'] - 12 if (n['midi'] - 12) in midis else (
                   n['midi'] - 24 if (n['midi'] - 24) in midis else None)
            if harm is not None:
                amp_n   = n.get('amplitude') or 1.0
                amp_low = max((m.get('amplitude') or 1.0) for m in g if m['midi'] == harm)
                if amp_n <= amp_low:
                    continue  # drop: octave harmonic of a louder lower note
            kept.append(n)
    return sorted(kept, key=lambda x: (x['start'], x['midi']))

# force a fresh detection (bypass the stale cache), then filter harmonics
for f in BASIC_PITCH_CACHE_DIR.glob("*_bp_notes.csv"):
    f.unlink()

bp = run_basic_pitch_notes(INPUT_AUDIO_PATH)
print("raw notes:", len(bp))
bp = drop_octave_harmonics(bp)
print("after harmonic removal:", len(bp))

key_label = "A minor"
chords   = detect_chords_from_notes(bp)
enriched = enrich_notes_with_context(bp, key_label, chords)
pred     = assign_caged_box(enriched, key=parse_key(key_label))
tempo, beats = estimate_beats_tempo(INPUT_AUDIO_PATH)
print(render_predicted_tab(pred, beats, tempo,
      title=f"CAGED-BOX TAB (harmonic-filtered) - key {key_label}", max_notes=200))

Running Basic Pitch on: HumanNature.mp3
Predicting MIDI for /content/drive/MyDrive/Capstone/Audio/HumanNature.mp3...
raw notes: 86
after harmonic removal: 83
CAGED-BOX TAB (harmonic-filtered) - key A minor
-----------------------------------------------
e|------------------------|------------------------|------------------------|------------------------|
B|------------------------|------------------------|------------------------|------------------------|
G|6-----------------------|------------------------|------------------------|7--6-----6--------------|
D|9-----------------------|7-----7-----------------|------------------9-----|---9--9-----9--7--9-----|
A|------------------------|---------9--7-----------|7-----5-----------------|------------------------|
E|5--------5-----5-----2--|------------------3-----|------------3-----------|------------------------|

e|------------------------|------------------------|------------------------|------------------------|
B|----------------------